# Lesson 26 Lab — Mixed-Bit Strategies and Sensitive-Layer Fallback

**Puzzle:** If only a few layers cause most quantization error, should every layer use more bits?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Uniform four-bit quantization spends the same precision on layers with different sensitivity. A mixed-bit policy measures how much each layer perturbs the end-to-end output, then allocates a fixed higher-precision budget to the worst offenders. The budget and the reassembled model result are as important as the ranking.


## 0. Predict before running

1. Predict which layers receive INT8 when only two fallbacks are allowed.
2. Compute the expected average bit width for two INT8 and four INT4 equal-size layers.
3. Explain why isolated layer sensitivity must be followed by an assembled-model evaluation.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Mixed-bit design assigns a precision/configuration to each layer or group under a memory, latency, and quality budget.

- Layer sensitivity is measured by the downstream objective under representative inputs.
- Mixed-bit allocation trades metadata and kernel diversity against quality.
- Fallback layers need a deterministic rule and a fixed memory budget.


## 2. Derive the mechanism

A sensitivity scan replaces one layer at a time and measures downstream change. A simple allocation then spends extra bits on the largest marginal quality benefit per added byte; interactions require re-evaluating the assembled model.

Let candidate bit assignment b_l minimize model error subject to `Σ n_l b_l / Σ n_l ≤ B`, where n_l is layer size and B is the average-bit budget. A simple greedy policy measures the output RMSE caused by quantizing one layer at a time and assigns extra precision to the largest scores. Interactions make this only a heuristic: two individually safe layers can amplify each other when quantized together.

Therefore the procedure has two stages—rank under a fixed probe, then assemble and retest the complete assignment. Storage, kernel compatibility, and latency must also be recalculated because mixed formats can add dispatch boundaries.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "26-mixed-bit-fallback"
device = require_cuda()
torch.manual_seed(2026 + 26)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | six-layer floating-point MLP and an all-INT4 candidate |
| Candidate | INT8 for the two most sensitive layers, INT4 for the remaining four |
| Held constant | equal layer sizes, calibration input, quantizers, two-layer fallback budget |
| Measurements | per-layer isolated RMSE, selected layers, average weight bits, assembled output error |
| Evidence | `pytorch-gpu` |

**Experiment:** Quantize a six-layer CUDA MLP one layer at a time, rank sensitivity, then construct a budgeted INT4/INT8 mixed-bit candidate.


## 5. Read the experiment code

The six-layer CUDA lab ranks INT4 substitutions, gives two layers INT8, computes average bits, and re-runs end to end.

The notebook quantizes each of six equal-size matrices to INT4 and INT8. It replaces one layer at a time to measure sensitivity against the full-precision network, selects the top two, constructs the mixed model, and re-evaluates end to end.

Because layers are equal size, the budget is transparent: `(2×8 + 4×4)/6 = 5.333` bits per weight. A real transformer would weight layers by parameter count and backend-compatible groupings.

Only after these variables match the protocol should the cell be executed.


In [2]:
dims=[256]*7; weights=[torch.randn(dims[i+1],dims[i],device=device)*0.05 for i in range(6)]; x=torch.randn(512,256,device=device)
def forward(ws):
    y=x
    for i,w in enumerate(ws): y=y@w.t(); y=torch.nn.functional.gelu(y) if i<5 else y
    return y
ref=forward(weights); q4=[]; q8=[]
for w in weights:
    q4.append(symmetric_quantize(w,bits=4,group_size=64)[2]); q8.append(symmetric_quantize(w,bits=8,group_size=64)[2])
sensitivity=[]
for i in range(6):
    ws=list(weights); ws[i]=q4[i]; sensitivity.append({"layer":i,"rmse":error_metrics(ref,forward(ws))["rmse"]})
fallback={r["layer"] for r in sorted(sensitivity,key=lambda z:z["rmse"],reverse=True)[:2]}; mixed=[q8[i] if i in fallback else q4[i] for i in range(6)]
bits=sum((8 if i in fallback else 4)*weights[i].numel() for i in range(6))/sum(w.numel() for w in weights)
result=base_result(26,"pytorch-gpu"); result.update({"layer_sensitivity":sensitivity,"int8_fallback_layers":sorted(fallback),
    "average_weight_bits":round(bits,3),"assembled_output_error":error_metrics(ref,forward(mixed)),
    "conclusion":"A budgeted mixed-bit candidate spent extra precision on measured sensitive layers and was re-evaluated end to end."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| INT8 fallback layers | 0, 1 |
| Average weight bits | 5.333 bits/weight |
| Layer 0 isolated RMSE | 0.001379 |
| Layer 1 isolated RMSE | 0.001304 |
| Assembled RMSE | 0.002484 |
| Assembled cosine | 0.976198 |


## 7. Interpret rather than merely print

Layers 0 and 1 had the highest isolated RMSE, 0.0013786 and 0.00130424, so they received INT8. The mixed assignment used 5.333 average bits and produced assembled RMSE 0.00248394 with cosine 0.976198.

The assembled error is larger than any isolated score, demonstrating interaction across layers. The ranking still gives a reproducible budgeted candidate, but whether it beats all-INT4 or another allocation must be judged with a frozen quality target and actual storage/runtime measurements.

**Inspection rule:** Compare the final end-to-end error and estimated storage, not only isolated layer rankings.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "assembled_output_error": {
    "cosine": 0.97619838,
    "mae": 0.0019622,
    "max_abs": 0.01424789,
    "rmse": 0.00248394
  },
  "average_weight_bits": 5.333,
  "conclusion": "A budgeted mixed-bit candidate spent extra precision on measured sensitive layers and was re-evaluated end to end.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:17+00:00",
  "int8_fallback_layers": [
    0,
    1
  ],
  "layer_sensitivity": [
    {
      "layer": 0,
      "rmse": 0.0013786
    },
    {
      "layer": 1,
      "rmse": 0.00130424
    },
    {
      "layer": 2,
      "rmse": 0.00120898
    },
    {
      "layer": 3,
      "rmse": 0.00125767
    },
    {
      "layer": 4,
      "rmse": 0.00121797
    },
    {
      "layer": 5,
      "rmse": 0.00121779
    }
  ],
  "

## 9. Make the bounded decision

> Use sensitivity scans to spend precision where it protects the objective, then re-measure the assembled model.

**Acceptance/rollback:** Freeze calibration/evaluation, record isolated sensitivities, budget, chosen fallback layers, final assembled quality, storage, operator coverage, and latency.

**Failure analysis:** Selecting fallback layers on the final task set overfits deployment evaluation. Comparing mixed-bit quality without reporting average bits is unfair. Backend fragmentation can also erase theoretical benefit if INT4 and INT8 layers use incompatible packing or force synchronization/materialization.


## 10. Extend the evidence

Add all-INT4 and all-INT8 assembled baselines, search several budgets, and plot quality versus effective bytes. Repeat sensitivity on multiple domains and sequence lengths. Then run a backend that supports the mixed formats and measure operator boundaries, memory, and latency.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
